# 1. Load Data

In [1]:
# Global Import and Directories
import sys
sys.path.append("..")
import Setup
from Setup import *

Current working directory: /home/qusta100/STGNN/02_Code/01_Preprocessing


In [2]:
# Path to the price data folder
folder_path = str(Setup.DATA_SCRATCH / "01_Original_Data" / "01_Prices")

# Find all CSV files matching the pattern "*-prices.csv"
csv_files = glob.glob(os.path.join(folder_path, "*-prices.csv"))

# Read each file and add it to the list
df_list = []
for file in csv_files:
    df_day = pd.read_csv(file, usecols=[0, 1, 2, 3, 4])
    df_day.columns = ["date", "station_uuid", "diesel", "e5", "e10"]
    df_list.append(df_day)

if not df_list:
    raise ValueError("Keine Dateien gefunden, die auf '*-prices.csv' passen.")

df_prices = pd.concat(df_list, ignore_index=True)

# Parse timestamps and remove timezone
df_prices["date"] = pd.to_datetime(df_prices["date"], utc=True).dt.tz_convert(None)

df_prices = df_prices.sort_values(["station_uuid", "date"])

# Get the overall time range (start & end)
start = df_prices["date"].min().replace(minute=0, second=0)
end = df_prices["date"].max().replace(minute=45, second=0)

# Create 15-minute time intervals
quarter_hours = pd.date_range(start=start, end=end, freq="15min")

In [3]:
# Load all station files from March and April
station_files = glob.glob(str(
    Setup.DATA_SCRATCH / "01_Original_Data" / "02_Stations" / "2025-0[3-5]-*-stations.csv"
))

stations_list = []
for file in station_files:
    df_station = pd.read_csv(file)
    # Extract the date from the filename and convert to datetime
    date_str = os.path.basename(file).split("-stations.csv")[0]
    df_station["day"] = pd.to_datetime(date_str)
    stations_list.append(df_station)

# Combine all station data into a single DataFrame
stations_all = pd.concat(stations_list, ignore_index=True)

In [4]:
# Merge Data Sets
left = df_prices.copy()
right = stations_all.copy()

left["date"] = pd.to_datetime(left["date"], errors="raise")
right["day"] = pd.to_datetime(right["day"], errors="raise")

right = right.rename(columns={"uuid": "station_uuid"})

left  = left.sort_values(["date", "station_uuid"], kind="mergesort").reset_index(drop=True)
right = right.sort_values(["day", "station_uuid"], kind="mergesort").reset_index(drop=True)

# 1) backward
bwd = pd.merge_asof(
    left, right,
    left_on="date", right_on="day",
    by="station_uuid",
    direction="backward",
    allow_exact_matches=True,
    suffixes=("", "_r")
)

# 2) forward
fwd = pd.merge_asof(
    left, right,
    left_on="date", right_on="day",
    by="station_uuid",
    direction="forward",
    allow_exact_matches=True,
    suffixes=("", "_r")
)

# Distanz in Tagen/Timedelta (NaT bleibt NaT)
bwd_dist = bwd["date"] - bwd["day"]
fwd_dist = fwd["day"] - fwd["date"]

# Welche Zeilen sollen forward nehmen?
use_fwd = bwd["day"].isna()


df = bwd.copy()
df.loc[use_fwd, :] = fwd.loc[use_fwd, :]

In [5]:
mask = df["date"] >= "2025-04-01"
df.loc[mask & df["day"].isna(), "station_uuid"].unique()

array(['4f4852cf-106d-400d-a76a-81515f0856c1'], dtype=object)

In [6]:
df.loc[mask].isna().sum()

date                      0
station_uuid              0
diesel                    0
e5                        0
e10                       0
name                     25
brand                270907
street                  404
house_number         494189
post_code               404
city                   1251
latitude                 22
longitude                22
first_active             22
openingtimes_json        22
day                      22
dtype: int64

In [7]:
# Filter for active stations
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["station_uuid", "date"])

# Difference to previous record
df["gap"] = df.groupby("station_uuid")["date"].diff()
cutoff = pd.Timedelta(days=15)
max_gap = df.groupby("station_uuid")["gap"].max()

active_stations = max_gap[max_gap <= cutoff].index
df = df[df["station_uuid"].isin(active_stations)]

# 2. Process Data

In [8]:
# Result list
resampled_list = []

# List of fuel price columns
price_cols = ["diesel", "e5", "e10"]

# List of metadata columns (everything except date, UUID, and prices)
metadata_cols = [col for col in df.columns if col not in ["date", "station_uuid"] + price_cols]

# For each station: resample time series and attach metadata
for station_id, group in df.groupby("station_uuid"):
    group = group.set_index("date").sort_index()

    # Resample prices (forward fill) for each fuel type
    reindexed_prices = {
        col: group[col].reindex(quarter_hours, method="ffill")
        for col in price_cols
    }

    # DataFrame with timestamps and UUID
    result = pd.DataFrame({
        "date": quarter_hours,
        "station_uuid": station_id,
        **{col: reindexed_prices[col].values for col in price_cols}
    })

    # Metadata from the first row (if available)
    meta = group[metadata_cols].iloc[0] if not group.empty else pd.Series(index=metadata_cols)

    # Attach metadata to each row
    for col in metadata_cols:
        result[col] = meta[col]

    resampled_list.append(result)

# Merge all resampled data
resampled_df = pd.concat(resampled_list, ignore_index=True)

# Restrict to April
resampled_df = resampled_df[resampled_df["date"] >= pd.Timestamp("2025-04-01 00:00:00")]

In [9]:
resampled_df["date"] = pd.to_datetime(resampled_df["date"])

# 1) Last observation date per station
last_seen = (
    resampled_df.groupby("station_uuid", as_index=False)["date"]
    .max()
    .rename(columns={"date": "last_seen"})
)

# 2) Create a new DataFrame
df_filtered = resampled_df.merge(last_seen, on="station_uuid", how="left")
df_filtered["last_seen"] = pd.to_datetime(df_filtered["last_seen"])

# 3) Moving activity
cutoff = pd.Timedelta(days=15)
mask_active = (df_filtered["date"] - df_filtered["last_seen"]) <= cutoff

df_filtered = df_filtered[mask_active].copy()

print("Obs after filter:", len(df_filtered))

Obs after filter: 48789536


In [10]:
# Replace 0 with NaN for all price columns
price_columns = ["diesel", "e5", "e10"]

for col in price_columns:
    if col in df_filtered.columns:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors="coerce")
        df_filtered.loc[df_filtered[col] <= 0, col] = np.nan

In [11]:
# Handling opening times
HOLIDAY_BIT = 1 << 7  # 128

def is_open(opening_json, timestamp, holiday: bool):
    try:
        if opening_json.strip() == "{}":
            return True

        data = json.loads(opening_json)

        # 1) Overrides haben Vorrang
        for ov in data.get("overrides", []):
            start = pd.to_datetime(ov["startp"])
            end = pd.to_datetime(ov["endp"])
            if start <= timestamp < end:
                return not ov.get("is_close", False)

        # 2) Regular openingTimes
        weekday = timestamp.weekday()      # 0=Mo ... 6=So
        mask = 1 << weekday                # 1,2,4,8,16,32,64
        if holiday:
            mask |= HOLIDAY_BIT             # +128

        time_str = timestamp.strftime("%H:%M")

        for block in data.get("openingTimes", []):
            if block.get("applicable_days", 0) & mask:
                for period in block.get("periods", []):
                    if period["startp"] <= time_str < period["endp"]:
                        return True

        return False
    except:
        return True
    
    
# Create a copy
df_clean_opening = df_filtered.copy()

# Holiday dummy (1 if date is 2025-04-18 or 2025-04-21, otherwise 0)
holidays = {pd.Timestamp('2025-04-18'), pd.Timestamp('2025-04-21')}

# .normalize() resets the time to midnight so you compare only dates
df_clean_opening['holiday'] = df_clean_opening['date'].dt.normalize().isin(holidays).astype(int)


# Price columns to set to NaN if station is closed
price_columns = [col for col in ["diesel", "e5", "e10"] if col in df_clean_opening.columns]


df_clean_opening["is_open"] = df_clean_opening.apply(
    lambda row: is_open(
        row["openingtimes_json"],
        row["date"],
        row["holiday"]
    ),
    axis=1
)

# Set all prices to NaN if closed
for col in price_columns:
    df_clean_opening.loc[~df_clean_opening["is_open"], col] = pd.NA

In [12]:
# Show data to validate function for opening times
df_clean_opening[df_clean_opening['is_open'] == False].head(5)

,date,station_uuid,diesel,e5,e10,name,brand,street,house_number,post_code,city,latitude,longitude,first_active,openingtimes_json,day,gap,last_seen,holiday,is_open
5744,2025-04-01 00:00:00,00041414-208c-4444-8888-acdc00000414,NaN,NaN,NaN,SELGROS,NaN,Oststrasse,17,40724,Hilden,51.169189,6.94895,2015-03-13 00:00:01+01,"{""openingTimes"":[{""applicable_days"":31,""period...",2025-03-28,NaT,2025-04-30 21:45:00,0,False
5745,2025-04-01 00:15:00,00041414-208c-4444-8888-acdc00000414,NaN,NaN,NaN,SELGROS,NaN,Oststrasse,17,40724,Hilden,51.169189,6.94895,2015-03-13 00:00:01+01,"{""openingTimes"":[{""applicable_days"":31,""period...",2025-03-28,NaT,2025-04-30 21:45:00,0,False
5746,2025-04-01 00:30:00,00041414-208c-4444-8888-acdc00000414,NaN,NaN,NaN,SELGROS,NaN,Oststrasse,17,40724,Hilden,51.169189,6.94895,2015-03-13 00:00:01+01,"{""openingTimes"":[{""applicable_days"":31,""period...",2025-03-28,NaT,2025-04-30 21:45:00,0,False
5747,2025-04-01 00:45:00,00041414-208c-4444-8888-acdc00000414,NaN,NaN,NaN,SELGROS,NaN,Oststrasse,17,40724,Hilden,51.169189,6.94895,2015-03-13 00:00:01+01,"{""openingTimes"":[{""applicable_days"":31,""period...",2025-03-28,NaT,2025-04-30 21:45:00,0,False
5748,2025-04-01 01:00:00,00041414-208c-4444-8888-acdc00000414,NaN,NaN,NaN,SELGROS,NaN,Oststrasse,17,40724,Hilden,51.169189,6.94895,2015-03-13 00:00:01+01,"{""openingTimes"":[{""applicable_days"":31,""period...",2025-03-28,NaT,2025-04-30 21:45:00,0,False


# 3. Adding more features

In [13]:
# Time/Day Features
df = df_clean_opening.copy()

# Assuming your DataFrame is named df and the timestamp column is 'date'
s = df['date'].astype(str) \
              .str.replace('\u00a0', ' ', regex=False) \
              .str.strip() \
              .str.replace(r'\s+', ' ', regex=True)
parsed_full  = pd.to_datetime(s, format='%Y-%m-%d %H:%M:%S', errors='coerce')
parsed_date  = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
df['date']   = parsed_full.fillna(parsed_date)

# 1) Weekday (Monday–Sunday)
df["weekday"] = df["date"].dt.weekday.astype(int)

# 2) One Hot Encodings
df["weekday"] = df["date"].dt.weekday
df = pd.get_dummies(df, columns=["weekday"], prefix="wd")

slot = (df["date"].dt.hour * 60 + df["date"].dt.minute) // 15
time_oh = pd.get_dummies(slot, prefix="t")
df = pd.concat([df, time_oh], axis=1)


# Example output
print(df[['date', 'holiday']].head())

                 date  holiday
0 2025-04-01 00:00:00        0
1 2025-04-01 00:15:00        0
2 2025-04-01 00:30:00        0
3 2025-04-01 00:45:00        0
4 2025-04-01 01:00:00        0


In [14]:
def clean_brand(df, brand_col="brand", new_col="brand_cat"):
    # create new variable
    df[new_col] = df[brand_col].copy()

    # 1) Direct renaming
    df.loc[df[new_col] == "AGIP ENI", new_col] = "Agip"
    df.loc[df[new_col] == "AVIA XPress", new_col] = "AVIA"
    df.loc[df[new_col] == "ORLEN Express", new_col] = "OLEN"
    df.loc[df[new_col] == "SB-Markttankstelle", new_col] = "SB"
    df.loc[df[new_col] == "TotalEnergies Truckstop", new_col] = "TotalEnergies"

    # 2) Starts-with rules (case-insensitive where useful)
    mask_hmh = df[new_col].str.startswith("Avia", na=False)
    df.loc[mask_hmh, new_col] = "Avia"

    mask_hmh = df[new_col].str.startswith("HMH MineralÃ¶l Service", na=False)
    df.loc[mask_hmh, new_col] = "HMH"

    mask_argos = df[new_col].str.startswith("Argos", na=False)
    df.loc[mask_argos, new_col] = "Argos"

    mask_allguth = df[new_col].str.startswith("Allguth", na=False)
    df.loc[mask_allguth, new_col] = "Allguth"

    mask_bergler = df[new_col].str.startswith("Bergler", na=False)
    df.loc[mask_bergler, new_col] = "Bergler"

    mask_bft_lower = df[new_col].str.startswith("bft", na=False)
    df.loc[mask_bft_lower, new_col] = "bft"

    mask_bft_upper = df[new_col].str.startswith("BFT", na=False)
    df.loc[mask_bft_upper, new_col] = "bft"

    mask_calpalm = df[new_col].str.startswith("Calpalm", na=False)
    df.loc[mask_calpalm, new_col] = "Calpalm"

    mask_lanfer = df[new_col].str.startswith("Lanfer", na=False)
    df.loc[mask_lanfer, new_col] = "Lanfer"

    mask_leu = df[new_col].str.startswith("Leu", na=False)
    df.loc[mask_leu, new_col] = "Leu"

    mask_markant = df[new_col].str.startswith("Markant", na=False)
    df.loc[mask_markant, new_col] = "Markant"

    mask_shell = df[new_col].str.startswith("SHELL", na=False)
    df.loc[mask_shell, new_col] = "Shell"

    mask_raiff = df[new_col].str.startswith("Raiffeisen", na=False)
    df.loc[mask_raiff, new_col] = "Raiffeisen"

    mask_frei = df[new_col].str.lower().str.startswith("frei", na=False)
    df.loc[mask_frei, new_col] = np.nan

    # 3) Remove all "supermarket" cases
    mask_supermarkt_tank = df[new_col].str.contains("Supermarkt-Tankstelle", na=False)
    mask_supermarkt = df[new_col].str.contains("Supermarkt", na=False)

    df.loc[mask_supermarkt_tank | mask_supermarkt, new_col] = np.nan

    # 4) Remove all brands that appear only once
    counts = df[new_col].value_counts(dropna=True)
    singletons = counts[counts == 1].index
    df.loc[df[new_col].isin(singletons), new_col] = np.nan
    
    df[new_col] = df[new_col].astype("category").cat.codes.astype(float)

    return df

df = clean_brand(df, brand_col="brand", new_col="brand_cat")

In [15]:
# Verification and tagging of highway stations using the stations dataset
autobahn_pattern = (
    r"\b(?:" 
    r"autobahn(?:\s*tankstelle)?"
    r"|autohof"
    r"|rasthof"
    r"|rast[-\s]?stätte"
    r"|raststaette"
    r"|tank\s*(?:und|&)\s*rast"
    r"|service\s*area"
    r"|an\s+der\s+a[0-9]{1,2}"
    r"|a\s*[0-9]{1,2}"
    r"|ausfahrt"
    r"|abfahrt"
    r"|bundesautobahn"
    r")\b"
)

# 1) Compute match per row in stations_all
row_is_highway = (
    stations_all["name"].str.contains(autobahn_pattern, case=False, na=False) |
    stations_all["street"].str.contains(autobahn_pattern, case=False, na=False)
)

# 2) Collapse to one flag per station uuid (any-day match -> True)
highway_by_uuid = (
    stations_all
    .assign(Highway_Identifier=row_is_highway)
    .groupby("uuid", as_index=True)["Highway_Identifier"]
    .any()
)

# 3) Add the flag to final_df (station_uuid refers to stations_all.uuid)
df["Highway_Identifier"] = df["station_uuid"].map(highway_by_uuid).fillna(False)

# 4) Output
print("Unique stations tagged as highway:", int(highway_by_uuid.sum()))
print("Rows flagged as highway:", int(df["Highway_Identifier"].sum()))

Unique stations tagged as highway: 452
Rows flagged as highway: 864472


/var/tmp/pbs.17055445.hpc-batch/ipykernel_63641/862861348.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Highway_Identifier"] = df["station_uuid"].map(highway_by_uuid).fillna(False)


In [ ]:
# Import raw oil prices via API
# Because of a missing network connection here, this step was run on a different server

import yfinance as yf

# 1. Retrieve hourly Brent data from April 1 to April 30, 2025
ticker = yf.Ticker("BZ=F")
df = ticker.history(start="2025-04-01", end="2025-05-01", interval="60m")
df = df.reset_index()
df['Datetime'] = df['Datetime'].dt.tz_localize(None)

# 2. Generate a complete 15-minute timestamp grid for April 2025
all_times = pd.date_range(
    start="2025-04-01 00:00:00",
    end="2025-04-30 23:45:00",
    freq="15min"
)
full_df = pd.DataFrame({'Datetime': all_times})

# 3. Select only the 'Open' price and rename it to 'Brent_Price'
df_short = df[['Datetime', 'Open']].rename(columns={'Open': 'Brent_Price'})

# 4. Merge the full 15-minute grid with the hourly prices,
#    filling each 15-min slot with the most recent (backward fill) hourly price
merged = pd.merge_asof(
    full_df.sort_values('Datetime'),
    df_short.sort_values('Datetime'),
    on='Datetime',
    direction='backward'
)

# 5. Save the result to a CSV file
merged.to_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "brent_april_2025_15min_filled.csv",
    index=False
)

# Optional: display the first 20 rows and total count (should be 2880 timestamps)
print(merged.head(20))
print("Total number of timestamps:", len(merged))

In [16]:
# Load data (restore from saved file)
brent = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "brent_april_2025_15min_filled.csv"
)
brent = brent.rename(columns={"Datetime": "date"})
brent['date'] = pd.to_datetime(brent['date'])

In [17]:
# Merge with main data set (df_main)
merged_df = pd.merge(df, brent, on='date', how='left')

# Show data set
merged_df.head()

,date,station_uuid,diesel,e5,e10,name,brand,street,house_number,post_code,...,t_89,t_90,t_91,t_92,t_93,t_94,t_95,brand_cat,Highway_Identifier,Brent_Price
0,2025-04-01 00:00:00,00006210-0037-4444-8888-acdc00006210,1.649,1.779,1.719,Beducker - Qualität günstig tanken,Beducker,Donauwörther Str.,47a,86405,...,False,False,False,False,False,False,False,115.0,False,74.959999
1,2025-04-01 00:15:00,00006210-0037-4444-8888-acdc00006210,1.649,1.779,1.719,Beducker - Qualität günstig tanken,Beducker,Donauwörther Str.,47a,86405,...,False,False,False,False,False,False,False,115.0,False,74.959999
2,2025-04-01 00:30:00,00006210-0037-4444-8888-acdc00006210,1.649,1.779,1.719,Beducker - Qualität günstig tanken,Beducker,Donauwörther Str.,47a,86405,...,False,False,False,False,False,False,False,115.0,False,74.959999
3,2025-04-01 00:45:00,00006210-0037-4444-8888-acdc00006210,1.649,1.779,1.719,Beducker - Qualität günstig tanken,Beducker,Donauwörther Str.,47a,86405,...,False,False,False,False,False,False,False,115.0,False,74.959999
4,2025-04-01 01:00:00,00006210-0037-4444-8888-acdc00006210,1.649,1.779,1.719,Beducker - Qualität günstig tanken,Beducker,Donauwörther Str.,47a,86405,...,False,False,False,False,False,False,False,115.0,False,74.959999


# 4. Sampling

## 4.1. Thurinigia

In [ ]:
# Load shape files
thuringia = gpd.read_file(
    Setup.DATA_SCRATCH / "03_Shapefiles" / "thuringia.geojson"
).to_crs("EPSG:4326")


# Clean coordinates
def to_scalar(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return v[0] if len(v) else np.nan
    return v

lon = pd.to_numeric(merged_df["longitude"].map(to_scalar), errors="coerce")
lat = pd.to_numeric(merged_df["latitude"].map(to_scalar),  errors="coerce")

# 2) Vectorize Coordinates
geom = gpd.points_from_xy(lon, lat)  # erwartet WGS84
gdf = gpd.GeoDataFrame(merged_df.copy(), geometry=geom, crs="EPSG:4326")

# Create Placeholder
gdf["in_thuringia"] = False
gdf["in_hesse"] = False
gdf["in_region_thuringia"] = False
gdf["in_region_hesse"] = False

# Spatial Join
joined = gpd.sjoin(gdf, thuringia[["geometry"]], how="left", predicate="within")
gdf["in_thuringia"] = joined.index_right.notna()

In [ ]:
# Initialize in_region: 1 for Thuringia, 0 otherwise
gdf["in_region_thuringia"] = gdf["in_thuringia"].fillna(False).astype(np.int8)

# Find neighbors within 30 km using BallTree (memory-efficient batch version)
valid = gdf["latitude"].notna() & gdf["longitude"].notna()
th_mask = gdf["in_thuringia"].fillna(False) & valid

if th_mask.any():
    lat_v = gdf.loc[valid, "latitude"].to_numpy(dtype=np.float32)
    lon_v = gdf.loc[valid, "longitude"].to_numpy(dtype=np.float32)
    coords_rad = np.deg2rad(np.column_stack([lat_v, lon_v])).astype(np.float32, copy=False)

    # Store the DataFrame indices of valid rows (to map results back later)
    valid_idx = gdf.index[valid].to_numpy()

    # Query points: only Thuringia stations
    th_lat = gdf.loc[th_mask, "latitude"].to_numpy(dtype=np.float32)
    th_lon = gdf.loc[th_mask, "longitude"].to_numpy(dtype=np.float32)
    th_coords_rad = np.deg2rad(np.column_stack([th_lat, th_lon])).astype(np.float32, copy=False)

    # Build BallTree using Haversine distance (spherical)
    tree = BallTree(coords_rad, metric="haversine")

    # Radius in radians (30 km on Earth’s surface)
    R_EARTH_KM = np.float32(6371.0088)
    radius_rad = np.float32(30.0) / R_EARTH_KM

    # Boolean mask for all valid rows; will be True for stations in the region
    region_mask_valid = np.zeros(coords_rad.shape[0], dtype=bool)

    # Batch query
    BATCH = 5000
    for start in range(0, th_coords_rad.shape[0], BATCH):
        end = start + BATCH
        neigh_lists = tree.query_radius(th_coords_rad[start:end], r=radius_rad)

        # Instead of concatenating large arrays, mark neighbors directly in the mask
        for arr in neigh_lists:
            if arr.size:
                region_mask_valid[arr] = True

    # Ensure all Thuringia points themselves are included
    region_mask_valid[np.searchsorted(valid_idx, gdf.index[th_mask])] = True

    # Map back to global DataFrame indices and set in_region_thuringia = 1
    gdf.loc[valid_idx[region_mask_valid], "in_region_thuringia"] = 1

# 6) Create a plain DataFrame without geometry
df_thuringia = pd.DataFrame(gdf).drop(columns="geometry")

In [22]:
# Number of entries in Thuringia
print(len(df_thuringia[df_thuringia["in_thuringia"] == True]))
print(len(df_thuringia[df_thuringia["in_region_thuringia"] == 1]))

# Filter only entries in the wanted region
df_thuringia = df_thuringia[df_thuringia["in_region_thuringia"]==1]

1171776
2909336


## 4.2. Hesse

In [18]:
# Load shape files
hesse = gpd.read_file(
    Setup.DATA_SCRATCH / "03_Shapefiles" / "hesse.geojson"
).to_crs("EPSG:4326")

# Clean coordinates
def to_scalar(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return v[0] if len(v) else np.nan
    return v

lon = pd.to_numeric(merged_df["longitude"].map(to_scalar), errors="coerce")
lat = pd.to_numeric(merged_df["latitude"].map(to_scalar),  errors="coerce")

# 2) Vectorize Coordinates
geom = gpd.points_from_xy(lon, lat)  # erwartet WGS84
gdf = gpd.GeoDataFrame(merged_df.copy(), geometry=geom, crs="EPSG:4326")

# Create Placeholder
gdf["in_thuringia"] = False
gdf["in_hesse"] = False
gdf["in_region_thuringia"] = False
gdf["in_region_hesse"] = False

# Spatial Join
joined = gpd.sjoin(gdf, hesse[["geometry"]], how="left", predicate="within")
gdf["in_hesse"] = joined.index_right.notna()

ERROR 1: PROJ: proj_create_from_database: Open of /gpfs/project/qusta100/pytorch/share/proj failed


In [ ]:
# Initialize in_region: 1 for Hessen, 0 otherwise
gdf["in_region_hesse"] = gdf["in_hesse"].fillna(False).astype(np.int8)

# Find neighbors within 30 km using BallTree (memory-efficient batch version)
valid = gdf["latitude"].notna() & gdf["longitude"].notna()
he_mask = gdf["in_hesse"].fillna(False) & valid

if he_mask.any():
    lat_v = gdf.loc[valid, "latitude"].to_numpy(dtype=np.float32)
    lon_v = gdf.loc[valid, "longitude"].to_numpy(dtype=np.float32)
    coords_rad = np.deg2rad(np.column_stack([lat_v, lon_v])).astype(np.float32, copy=False)

    # Store the DataFrame indices of valid rows (to map results back later)
    valid_idx = gdf.index[valid].to_numpy()

    # Query points: only Hesse stations
    th_lat = gdf.loc[he_mask, "latitude"].to_numpy(dtype=np.float32)
    th_lon = gdf.loc[he_mask, "longitude"].to_numpy(dtype=np.float32)
    th_coords_rad = np.deg2rad(np.column_stack([th_lat, th_lon])).astype(np.float32, copy=False)

    # Build BallTree using Haversine distance (spherical)
    tree = BallTree(coords_rad, metric="haversine")

    # Radius in radians (30 km on Earth’s surface)
    R_EARTH_KM = np.float32(6371.0088)
    radius_rad = np.float32(30.0) / R_EARTH_KM

    # Boolean mask for all valid rows; will be True for stations in the region
    region_mask_valid = np.zeros(coords_rad.shape[0], dtype=bool)

    # Batch query
    BATCH = 5000
    for start in range(0, th_coords_rad.shape[0], BATCH):
        end = start + BATCH
        neigh_lists = tree.query_radius(th_coords_rad[start:end], r=radius_rad)

        # Instead of concatenating large arrays, mark neighbors directly in the mask
        for arr in neigh_lists:
            if arr.size:
                region_mask_valid[arr] = True

    # Ensure all Hessse points themselves are included
    region_mask_valid[np.searchsorted(valid_idx, gdf.index[he_mask])] = True

    # Map back to global DataFrame indices and set in_region_hesse = 1
    gdf.loc[valid_idx[region_mask_valid], "in_region_hesse"] = 1

# 6) Create a plain DataFrame without geometry
df_hesse = pd.DataFrame(gdf).drop(columns="geometry")

In [ ]:
# Number of entries in Hesse
print(len(df_hesse[df_hesse["in_hesse"] == True]))
print(len(df_hesse[df_hesse["in_region_hesse"] == 1]))

# Filter only entries in the wanted region
df_hesse = df_hesse[df_hesse["in_region_hesse"]==1]

# 5. Save the files

In [ ]:
# Thuringia, with highways
df_thuringia.to_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Thuringia.csv",
    index=False
)

In [ ]:
# Thuringia, w/o highways
df_thuringia_no_highway = df_thuringia[df_thuringia["Highway_Identifier"] == False].copy()

df_thuringia_no_highway.to_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Thuringia_no_highways.csv",
    index=False
)

In [ ]:
# Hesse, with highways
df_hesse.to_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Hesse.csv",
    index=False
)

In [ ]:
# Hesse, w/o highways
df_hesse_no_highway = df_hesse[df_hesse["Highway_Identifier"] == False].copy()

df_hesse.to_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Hesse_no_highways.csv",
    index=False
)